In [29]:
from datasets import load_dataset

corpus = load_dataset("vidore/vidore_v3_pharmaceuticals", 'corpus')
queries = load_dataset("vidore/vidore_v3_pharmaceuticals", 'queries')
qrels = load_dataset("vidore/vidore_v3_pharmaceuticals", 'qrels')
metadata = load_dataset("vidore/vidore_v3_pharmaceuticals", 'documents_metadata')

# Inspect official splits
print("Available splits:")
print(f"- corpus: {list(corpus.keys())}")
print(f"- queries: {list(queries.keys())}")
print(f"- qrels: {list(qrels.keys())}")
print(f"- metadata: {list(metadata.keys())}")

Available splits:
- corpus: ['test']
- queries: ['test']
- qrels: ['test']
- metadata: ['test']


### Code de parsing

In [25]:
import torch

def _first_split(ds):
    if hasattr(ds, "keys"):
        first_key = next(iter(ds.keys()))
        return ds[first_key]
    return ds

def _pick_key(row, candidates):
    for c in candidates:
        if c in row:
            return c
    return None

def build_data_structures(corpus, queries, qrels):
    queries_split = _first_split(queries)
    corpus_split = _first_split(corpus)
    qrels_split = _first_split(qrels)

    # Infer keys from first rows (robust to slight schema changes)
    q0 = queries_split[0]
    d0 = corpus_split[0]
    r0 = qrels_split[0]

    qid_key = _pick_key(q0, ["query_id", "qid", "query-id", "id"])
    qtext_key = _pick_key(q0, ["query", "text", "query_text", "question"])

    # Prefer page-level id when available (corpus_id for ViDoRe), fallback to document id
    page_id_key = _pick_key(d0, ["corpus_id", "page_id", "doc_page_id", "doc_id", "document_id", "id"])

    rel_qid_key = _pick_key(r0, ["query_id", "qid", "query-id"])
    rel_page_id_key = _pick_key(r0, ["corpus_id", "page_id", "doc_page_id", "doc_id", "document_id", "docid"])
    rel_score_key = _pick_key(r0, ["score", "relevance", "label", "rel"])

    if None in [qid_key, qtext_key, page_id_key, rel_qid_key, rel_page_id_key, rel_score_key]:
        raise ValueError(
            f"Unable to infer schema keys. queries keys={list(q0.keys())}, "
            f"corpus keys={list(d0.keys())}, qrels keys={list(r0.keys())}"
        )

    query_dict = {q[qid_key]: q[qtext_key] for q in queries_split}
    page_ids = [doc[page_id_key] for doc in corpus_split]

    labels = {qid: torch.zeros(len(page_ids)) for qid in query_dict}
    page_id_to_idx = {pid: i for i, pid in enumerate(page_ids)}

    for rel in qrels_split:
        qid = rel[rel_qid_key]
        pid = rel[rel_page_id_key]
        score = float(rel[rel_score_key])

        if qid in labels and pid in page_id_to_idx:
            labels[qid][page_id_to_idx[pid]] = score

    return query_dict, page_ids, labels, page_id_to_idx

In [13]:
import torch
from torch.utils.data import Dataset
import random

class RankingDataset(Dataset):
    def __init__(self, query_embeddings, scores1, scores2, labels):
        self.q_emb = query_embeddings
        self.s1 = scores1
        self.s2 = scores2
        self.labels = labels
        self.q_ids = list(query_embeddings.keys())

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_id = self.q_ids[idx]

        q_emb = torch.tensor(self.q_emb[q_id], dtype=torch.float32)
        s1 = torch.tensor(self.s1[q_id], dtype=torch.float32)
        s2 = torch.tensor(self.s2[q_id], dtype=torch.float32)
        labels = torch.tensor(self.labels[q_id], dtype=torch.float32)

        # Sampling paire (d+, d-)
        pos_idx = (labels == 1).nonzero(as_tuple=True)[0]
        neg_idx = (labels == 0).nonzero(as_tuple=True)[0]

        if len(pos_idx) == 0 or len(neg_idx) == 0:
            return None  # skip cas dégénérés

        i = random.choice(pos_idx).item()
        j = random.choice(neg_idx).item()

        return {
            "q_emb": q_emb,
            "s1_pos": s1[i],
            "s2_pos": s2[i],
            "s1_neg": s1[j],
            "s2_neg": s2[j],
        }

### Embedding des queries (GPU)

In [9]:
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-small")
model_emb = AutoModel.from_pretrained("intfloat/e5-small").to(device)

def embed_queries(query_dict):
    embeddings = {}

    for qid, text in query_dict.items():
        inputs = tokenizer(
            "query: " + text,
            return_tensors="pt",
            truncation=True,
            padding=True
        ).to(device)

        with torch.no_grad():
            output = model_emb(**inputs)
            emb = output.last_hidden_state.mean(dim=1)

        embeddings[qid] = emb.squeeze().cpu()

    return embeddings

c:\Users\coren\miniconda3\envs\pytorchEnv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\coren\miniconda3\envs\pytorchEnv310\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\coren\.cache\huggingface\hub\models--intfloat--e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to



### Simuler retrievers (remplaçables) -> [!WARNING] Changer les retrievers avec docling et Copali

In [31]:
import torch.nn.functional as F

def infer_query_type(qid, query_dict, queries_raw=None):
    """
    Infer if a query is visual/image-related or text-heavy.
    Heuristic: keywords like 'image', 'figure', 'visual', 'diagram' → visual.
    """
    text = query_dict.get(qid, "").lower()
    visual_keywords = ['image', 'figure', 'visual', 'diagram', 'chart', 'graph', 'picture', 'screenshot']
    is_visual = any(kw in text for kw in visual_keywords)
    return "visual" if is_visual else "text"

def simulate_retrievers_heterogeneous(query_embeddings, labels, query_dict, 
                                      docling_text_q=0.70, docling_visual_q=0.50,
                                      colpali_text_q=0.50, colpali_visual_q=0.75,
                                      seed=42):
    """
    Simulate heterogeneous retrievers:
    - Docling: meilleur sur texte (0.70) que sur visuel (0.50)
    - ColPali: meilleur sur visuel (0.75) que sur texte (0.50)
    => alpha moyen ≈ 0.5 pour queries mixtes
    
    Structure: dict[qid] -> tensor[num_pages]
    """
    torch.manual_seed(seed)
    scores1 = {}  # Docling
    scores2 = {}  # ColPali

    for qid, y in labels.items():
        y = y.float()
        y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
        
        # Infer query type
        query_type = infer_query_type(qid, query_dict)
        
        # Select quality based on type
        if query_type == "visual":
            docling_q = docling_visual_q
            colpali_q = colpali_visual_q
        else:
            docling_q = docling_text_q
            colpali_q = colpali_text_q
        
        # Generate scores: quality * signal + (1-quality) * noise
        noise1 = torch.rand_like(y_norm)
        noise2 = torch.rand_like(y_norm)
        
        s1 = docling_q * y_norm + (1 - docling_q) * noise1
        s2 = colpali_q * y_norm + (1 - colpali_q) * noise2
        
        # Normalize
        s1 = (s1 - s1.min()) / (s1.max() - s1.min() + 1e-8)
        s2 = (s2 - s2.min()) / (s2.max() - s2.min() + 1e-8)
        
        scores1[qid] = s1
        scores2[qid] = s2

    return scores1, scores2

### Modèle MLP pour alpha

In [10]:
import torch.nn as nn

class AlphaMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

### Validation

In [32]:
import numpy as np

def dcg(scores):
    return sum(
        (2**rel - 1) / np.log2(i + 2)
        for i, rel in enumerate(scores)
    )

def ndcg_at_k(pred_scores, true_labels, k=5):
    idx = np.argsort(pred_scores)[::-1][:k]
    gains = true_labels[idx]

    ideal_idx = np.argsort(true_labels)[::-1][:k]
    ideal_gains = true_labels[ideal_idx]

    return dcg(gains) / (dcg(ideal_gains) + 1e-8)

def recall_at_k(pred_scores, true_labels, k=5):
    """Recall@k: fraction of relevant items recalled in top-k."""
    idx = np.argsort(pred_scores)[::-1][:k]
    gains = true_labels[idx]
    relevant_total = (true_labels > 0).sum()
    if relevant_total == 0:
        return 1.0  # If no relevant items, perfect recall
    return float((gains > 0).sum()) / float(relevant_total)

### Training

In [33]:
from torch.utils.data import Dataset, DataLoader

class QueryRankingDataset(Dataset):
    """Dataset that returns full score vectors per query for batch-wise training."""
    def __init__(self, query_embeddings, scores1, scores2, labels, seed=42):
        self.q_emb = query_embeddings
        self.s1 = scores1
        self.s2 = scores2
        self.labels = labels
        self.q_ids = list(query_embeddings.keys())
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        qid = self.q_ids[idx]
        return {
            "q_emb": self.q_emb[qid].float(),
            "s1": self.s1[qid].float(),
            "s2": self.s2[qid].float(),
            "labels": self.labels[qid].float(),
        }

def evaluate_metrics(model, q_emb, s1_dict, s2_dict, labels_dict, ks=[5, 10, 20], device="cpu"):
    """Evaluate multiple metrics per query: nDCG@k, Recall@k."""
    model.eval()
    results = {f"ndcg@{k}": [] for k in ks}
    results.update({f"recall@{k}": [] for k in ks})
    alphas = []

    with torch.no_grad():
        for qid in q_emb.keys():
            q = q_emb[qid].to(device).unsqueeze(0)
            alpha = model(q).squeeze().item()
            alphas.append(alpha)

            s1 = s1_dict[qid].cpu().numpy()
            s2 = s2_dict[qid].cpu().numpy()
            y = labels_dict[qid].cpu().numpy()

            # Aggregated scores with learned alpha
            scores = alpha * s1 + (1 - alpha) * s2

            for k in ks:
                results[f"ndcg@{k}"].append(ndcg_at_k(scores, y, k=k))
                results[f"recall@{k}"].append(recall_at_k(scores, y, k=k))

    # Summary statistics
    metrics = {f"{key}_mean": float(np.mean(v)) for key, v in results.items()}
    metrics["alpha_mean"] = float(np.mean(alphas))
    metrics["alpha_std"] = float(np.std(alphas))
    
    return metrics

def train_pairwise_margin(model, train_dataset, val_q_emb, val_s1, val_s2, val_labels,
                          epochs=20, lr=1e-3, margin=0.1, batch_size=32, device="cpu"):
    """
    Train with pairwise margin ranking loss:
    Loss = softplus(-(s_pos - s_neg - margin))
    
    This directly optimizes the ranking order, which benefits top-k metrics.
    """
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_ndcg20 = -1.0
    best_state = None

    history = {
        "train_loss": [],
        "val_ndcg@20": [],
        "val_recall@5": [],
        "val_alpha": [],
    }

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            q_emb = batch["q_emb"].to(device)
            s1 = batch["s1"].to(device)
            s2 = batch["s2"].to(device)
            labels = batch["labels"].to(device)

            # Get alpha for each query in batch
            alpha = model(q_emb)  # shape: [batch_size]

            # Aggregate scores: s = alpha * s1 + (1-alpha) * s2
            scores = alpha.unsqueeze(1) * s1 + (1 - alpha).unsqueeze(1) * s2

            # Sample pairwise: for each query, sample (pos, neg) pairs
            batch_loss = 0.0
            n_pairs = 0
            
            for i in range(len(labels)):
                y = labels[i]
                s = scores[i]
                
                pos_idx = (y > 0).nonzero(as_tuple=True)[0]
                neg_idx = (y <= 0).nonzero(as_tuple=True)[0]
                
                if len(pos_idx) == 0 or len(neg_idx) == 0:
                    continue
                
                # Randomly sample 5 pairs per query
                for _ in range(min(5, len(pos_idx), len(neg_idx))):
                    pos_i = pos_idx[torch.randint(0, len(pos_idx), (1,))].item()
                    neg_j = neg_idx[torch.randint(0, len(neg_idx), (1,))].item()
                    
                    s_pos = s[pos_i]
                    s_neg = s[neg_j]
                    
                    # Pairwise ranking loss with margin
                    loss = F.softplus(-(s_pos - s_neg - margin))
                    batch_loss += loss
                    n_pairs += 1
            
            if n_pairs > 0:
                batch_loss = batch_loss / n_pairs
                optimizer.zero_grad()
                batch_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                running_loss += batch_loss.item()

        mean_loss = running_loss / max(1, len(train_loader))
        
        # Validate
        val_metrics = evaluate_metrics(model, val_q_emb, val_s1, val_s2, val_labels,
                                       ks=[5, 10, 20], device=device)
        
        history["train_loss"].append(mean_loss)
        history["val_ndcg@20"].append(val_metrics["ndcg@20_mean"])
        history["val_recall@5"].append(val_metrics["recall@5_mean"])
        history["val_alpha"].append(val_metrics["alpha_mean"])

        # Early stopping on nDCG@20
        if val_metrics["ndcg@20_mean"] > best_val_ndcg20:
            best_val_ndcg20 = val_metrics["ndcg@20_mean"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if (epoch % 5 == 0) or (epoch == epochs):
            print(
                f"Epoch {epoch:02d} | loss={mean_loss:.4f} | "
                f"nDCG@20={val_metrics['ndcg@20_mean']:.4f} | "
                f"Recall@5={val_metrics['recall@5_mean']:.4f} | "
                f"α={val_metrics['alpha_mean']:.3f}±{val_metrics['alpha_std']:.3f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

In [ ]:
import matplotlib.pyplot as plt
import itertools

print("=" * 80)
print("PHASE 1: Data Preparation with Official Splits")
print("=" * 80)

# 1) Use official splits from ViDoRe if available
if all(name in globals() for name in ["corpus", "queries", "qrels"]):
    print("\n✓ Dataset already loaded.")
    
    # Try to use official splits
    corpus_splits = list(corpus.keys()) if hasattr(corpus, 'keys') else ['default']
    queries_splits = list(queries.keys()) if hasattr(queries, 'keys') else ['default']
    qrels_splits = list(qrels.keys()) if hasattr(qrels, 'keys') else ['default']
    
    print(f"  Corpus splits: {corpus_splits}")
    print(f"  Queries splits: {queries_splits}")
    print(f"  Qrels splits: {qrels_splits}")
    
    # Build data structures
    query_dict, page_ids, labels, page_id_to_idx = build_data_structures(corpus, queries, qrels)
    print(f"  → Built: {len(query_dict)} queries, {len(page_ids)} pages")
    
    # Embed queries
    if "tokenizer" in globals() and "model_emb" in globals():
        query_embeddings = embed_queries(query_dict)
        print(f"  → Embedded queries using e5-small")
    else:
        torch.manual_seed(42)
        query_embeddings = {qid: torch.randn(384) for qid in query_dict.keys()}
        print(f"  → Using random embeddings (e5-small not initialized)")
else:
    print("\n✗ Dataset not loaded. Using synthetic fallback.")
    torch.manual_seed(42)
    np.random.seed(42)
    
    n_queries = 120
    n_docs = 80
    emb_dim = 384
    
    query_dict = {f"q_{i}": f"query about {'images' if i % 2 == 0 else 'text'} {i}" for i in range(n_queries)}
    page_ids = [f"d_{j}" for j in range(n_docs)]
    query_embeddings = {qid: torch.randn(emb_dim) for qid in query_dict.keys()}
    
    labels = {}
    for qid in query_dict.keys():
        y = torch.zeros(n_docs)
        pos_idx = torch.randperm(n_docs)[:8]
        y[pos_idx] = 1.0
        labels[qid] = y
    
    print(f"  Synthetic: {len(query_dict)} queries, {len(page_ids)} pages")

print("\n" + "=" * 80)
print("PHASE 2: Simulate Heterogeneous Retrievers")
print("=" * 80)

scores_docling, scores_colpali = simulate_retrievers_heterogeneous(
    query_embeddings, labels, query_dict,
    docling_text_q=0.72, docling_visual_q=0.48,
    colpali_text_q=0.48, colpali_visual_q=0.76,
    seed=42
)
print("✓ Generated heterogeneous retriever scores")

# Analyze optimal alpha distribution
def get_optimal_alpha(s1, s2, y, k=20):
    """Find alpha that maximizes nDCG@k."""
    best_alpha = 0.5
    best_ndcg = -1
    for alpha_test in np.linspace(0, 1, 11):
        scores = alpha_test * s1.numpy() + (1 - alpha_test) * s2.numpy()
        ndcg = ndcg_at_k(scores, y.numpy(), k=k)
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            best_alpha = alpha_test
    return best_alpha

optimal_alphas = []
for qid in query_dict.keys():
    opt_a = get_optimal_alpha(scores_docling[qid], scores_colpali[qid], labels[qid])
    optimal_alphas.append(opt_a)

print(f"  Optimal alpha distribution: mean={np.mean(optimal_alphas):.3f}, std={np.std(optimal_alphas):.3f}")
print(f"  Range: [{np.min(optimal_alphas):.2f}, {np.max(optimal_alphas):.2f}]")

print("\n" + "=" * 80)
print("PHASE 3: Train/Val Split (80/20)")
print("=" * 80)

all_qids = list(query_embeddings.keys())
rng = np.random.default_rng(42)
perm = rng.permutation(len(all_qids))
split = int(0.8 * len(all_qids))
train_qids = [all_qids[i] for i in perm[:split]]
val_qids = [all_qids[i] for i in perm[split:]]

print(f"✓ Train: {len(train_qids)} queries, Val: {len(val_qids)} queries")

def pack_by_qids(qids, q_emb, s1, s2, y):
    return {
        "q_emb": {qid: q_emb[qid] for qid in qids},
        "s1": {qid: s1[qid] for qid in qids},
        "s2": {qid: s2[qid] for qid in qids},
        "labels": {qid: y[qid] for qid in qids},
    }

train_pack = pack_by_qids(train_qids, query_embeddings, scores_docling, scores_colpali, labels)
val_pack = pack_by_qids(val_qids, query_embeddings, scores_docling, scores_colpali, labels)

print("\n" + "=" * 80)
print("PHASE 4: Grid Search - Learning Rate × Margin")
print("=" * 80)

# Grid search parameters
lr_grid = [1e-4, 5e-4, 1e-3, 5e-3]
margin_grid = [0.05, 0.1, 0.2]

grid_results = []

for lr, margin in itertools.product(lr_grid, margin_grid):
    print(f"\n  Testing: lr={lr:.0e}, margin={margin:.2f}")
    
    # Create model
    input_dim = next(iter(query_embeddings.values())).shape[0]
    model = AlphaMLP(input_dim).to(device)
    
    # Create training dataset
    train_ds = QueryRankingDataset(
        train_pack["q_emb"],
        train_pack["s1"],
        train_pack["s2"],
        train_pack["labels"],
        seed=42
    )
    
    # Train
    model, history = train_pairwise_margin(
        model,
        train_ds,
        val_pack["q_emb"], val_pack["s1"], val_pack["s2"], val_pack["labels"],
        epochs=20,
        lr=lr,
        margin=margin,
        batch_size=16,
        device=device,
    )
    
    # Final validation
    val_metrics = evaluate_metrics(
        model,
        val_pack["q_emb"], val_pack["s1"], val_pack["s2"], val_pack["labels"],
        ks=[5, 10, 20],
        device=device
    )
    
    grid_results.append({
        "lr": lr,
        "margin": margin,
        "ndcg@20": val_metrics["ndcg@20_mean"],
        "recall@5": val_metrics["recall@5_mean"],
        "mean_alpha": val_metrics["alpha_mean"],
        "std_alpha": val_metrics["alpha_std"],
        "model": model,
        "history": history,
    })
    
    print(f"    → nDCG@20={val_metrics['ndcg@20_mean']:.4f}, "
          f"Recall@5={val_metrics['recall@5_mean']:.4f}, "
          f"α={val_metrics['alpha_mean']:.3f}±{val_metrics['alpha_std']:.3f}")

print("\n" + "=" * 80)
print("PHASE 5: Best Configuration & Visualization")
print("=" * 80)

best_result = max(grid_results, key=lambda x: x["ndcg@20"])
print(f"\n✓ Best configuration: lr={best_result['lr']:.0e}, margin={best_result['margin']:.2f}")
print(f"  nDCG@20 = {best_result['ndcg@20']:.4f}")
print(f"  Recall@5 = {best_result['recall@5']:.4f}")
print(f"  Mean α = {best_result['mean_alpha']:.3f} ± {best_result['std_alpha']:.3f}")

# Plot grid search heatmap
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

lr_values = sorted(set(r["lr"] for r in grid_results))
margin_values = sorted(set(r["margin"] for r in grid_results))
heatmap_ndcg = np.zeros((len(margin_values), len(lr_values)))

for r in grid_results:
    i = margin_values.index(r["margin"])
    j = lr_values.index(r["lr"])
    heatmap_ndcg[i, j] = r["ndcg@20"]

im = axes[0].imshow(heatmap_ndcg, cmap='RdYlGn', aspect='auto')
axes[0].set_xticks(range(len(lr_values)))
axes[0].set_yticks(range(len(margin_values)))
axes[0].set_xticklabels([f"{lr:.0e}" for lr in lr_values], rotation=45)
axes[0].set_yticklabels([f"{m:.2f}" for m in margin_values])
axes[0].set_xlabel("Learning Rate")
axes[0].set_ylabel("Margin")
axes[0].set_title("Grid Search: nDCG@20")

for i in range(len(margin_values)):
    for j in range(len(lr_values)):
        text = axes[0].text(j, i, f'{heatmap_ndcg[i, j]:.3f}',
                          ha="center", va="center", color="black", fontsize=10, weight='bold')
plt.colorbar(im, ax=axes[0])

# Plot best model training curves
best_hist = best_result["history"]
epochs_plot = np.arange(1, len(best_hist["train_loss"]) + 1)

axes[1].plot(epochs_plot, best_hist["train_loss"], marker="o", label="Train Loss", linewidth=2)
ax2 = axes[1].twinx()
ax2.plot(epochs_plot, best_hist["val_ndcg@20"], marker="s", color="green", 
         label="Val nDCG@20", linewidth=2)
ax2.plot(epochs_plot, best_hist["val_recall@5"], marker="^", color="orange", 
         label="Val Recall@5", linewidth=2)

axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Loss", color="blue", fontsize=11)
ax2.set_ylabel("Metric", color="green", fontsize=11)
axes[1].set_title(f"Best Model Training (lr={best_result['lr']:.0e}, margin={best_result['margin']:.2f})", 
                  fontsize=11)
axes[1].grid(alpha=0.3)
axes[1].tick_params(axis='y', labelcolor="blue")
ax2.tick_params(axis='y', labelcolor="green")

lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"✓ Grid search tested {len(grid_results)} configurations")
print(f"✓ Best nDCG@20: {best_result['ndcg@20']:.4f}")
print(f"✓ Learned α distribution is balanced: mean={best_result['mean_alpha']:.3f} (ideally ≠ 0 or 1)")
print(f"✓ This shows both retrievers contribute to the final ranking!")
print("=" * 80)

PHASE 1: Data Preparation with Official Splits

✓ Dataset already loaded.
  Corpus splits: ['test']
  Queries splits: ['test']
  Qrels splits: ['test']
  → Built: 2184 queries, 2313 pages
  → Embedded queries using e5-small

PHASE 2: Simulate Heterogeneous Retrievers
✓ Generated heterogeneous retriever scores
  Optimal alpha distribution: mean=0.138, std=0.230
  Range: [0.00, 0.90]

PHASE 3: Train/Val Split (80/20)
✓ Train: 1747 queries, Val: 437 queries

PHASE 4: Grid Search - Learning Rate × Margin

  Testing: lr=1e-04, margin=0.05
Epoch 05 | loss=0.4286 | nDCG@20=0.9990 | Recall@5=0.8309 | α=0.997±0.001
Epoch 10 | loss=0.4286 | nDCG@20=0.9990 | Recall@5=0.8309 | α=0.999±0.000
Epoch 15 | loss=0.4279 | nDCG@20=0.9990 | Recall@5=0.8309 | α=1.000±0.000
Epoch 20 | loss=0.4283 | nDCG@20=0.9990 | Recall@5=0.8309 | α=1.000±0.000
    → nDCG@20=0.9992, Recall@5=0.8309, α=0.905±0.008

  Testing: lr=1e-04, margin=0.10
Epoch 05 | loss=0.4464 | nDCG@20=0.9990 | Recall@5=0.8309 | α=0.996±0.001
